# LG-02: Tools 深度掌握

> **阶段**: LG-02 | **难度**: 中级 | **预计时长**: 3-4 小时

## 学习目标
- 用书房预约助手串起 Tool 定义、调用、执行和复用
- 先用最普通的 Tool 调用跑出上下文爆炸，再引入裁剪和 runtime 旁路透传
- 跑通一次本地 `ToolNode` 调用，看到工具结果如何进入消息流
- 理解 ReAct Agent 与 `bind_tools` 的区别
- 区分 Tool 直接作为节点、`ToolNode`、`create_agent` 的使用边界


In [ ]:
# 安装依赖（如未安装请取消注释）
# !pip install -U langgraph langchain langchain-core langchain-openai pydantic


## 1. 核心概念：Tool 是 Agent 连接外部能力的接口

Tool 把数据库、业务系统、本地文件、HTTP API 等外部能力封装成可调用函数，让 Agent 可以在对话过程中读取事实、执行动作，并把结果继续交给后续节点或模型。

本节要区分三件事：

1. `Tool`：定义一个可被调用的外部能力。
2. `ToolNode`：在 LangGraph 图里执行模型发出的 `tool_calls`。
3. `Agent`：决定何时调用工具、如何阅读工具结果、如何继续回复用户。

先思考一个问题：如果查询接口一次返回几万条候选，最直接的做法是把完整 JSON 返回给 Agent。这个做法能跑通第一步，但下一次模型调用会发生什么？


## 2. 案例背景：书房预约助手

本节实现一个 `StudyRoomBookingBot`。用户会连续提出这些请求：

- `帮我查一下明天下午平原轩还有没有空位`
- `把 15:00 之后能约的时段列出来`
- `那就帮我约 16:00-18:00`
- `看看我最近的预约记录`

动手前先判断这条链路需要哪些部分：查询规则、查询空位、缓存、提交预约、查询历史、上下文管理。再进一步想：哪些数据应该给 LLM 看，哪些数据只应该保留在程序内部？

本节使用本地脱敏 JSON 模拟业务后端，再用 Tool、ToolNode 和可选 ReAct Agent 串起来。


## 3. 本地 JSON 数据源

真实系统里，Tool 通常会访问数据库、RPC 或 HTTP API。这里先用本地 JSON 代替后端，让所有代码可以在 notebook 里直接运行。

数据已经在进入 JSON 前完成脱敏。案例代码只消费脱敏后的本地数据，不在 notebook 里演示脱敏处理。


In [ ]:
from pathlib import Path
import json
import os
from pprint import pprint
from typing import Any, Annotated, Type

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import BaseTool, StructuredTool, tool

DATA_PATH = Path("study_room_data.json")
if not DATA_PATH.exists():
    DATA_PATH = Path("turtorial/LG-02-tools/study_room_data.json")

BOOKING_DATA = json.loads(DATA_PATH.read_text(encoding="utf-8"))


def json_chars(value: Any) -> int:
    return len(json.dumps(value, ensure_ascii=False))


def collect_availability_candidates(
    room_name: str | None,
    booking_date: str,
    start_after: str | None = None,
) -> list[dict]:
    items = []
    for room in BOOKING_DATA["rooms"]:
        if room_name and room["room_name"] != room_name:
            continue
        for area in room["areas"]:
            for seat in area["seats"]:
                for slot in seat["slots"]:
                    if start_after and slot[:5] < start_after:
                        continue
                    start_time, end_time = slot.split("-")
                    items.append({
                        "candidate_id": f"{room['room_id']}_{area['area_id']}_{seat['seat_id']}_{start_time}",
                        "room_id": room["room_id"],
                        "room_name": room["room_name"],
                        "campus": room["campus"],
                        "rules": room["rules"],
                        "area_id": area["area_id"],
                        "area_name": area["area_name"],
                        "noise_level": area["noise_level"],
                        "seat_id": seat["seat_id"],
                        "seat_name": seat["seat_name"],
                        "floor": seat["floor"],
                        "near_power": seat["near_power"],
                        "window_side": seat["window_side"],
                        "capacity": seat["capacity"],
                        "booking_date": booking_date,
                        "start_time": start_time,
                        "end_time": end_time,
                        "source": "local_long_json",
                    })
    return items


sample_candidates = collect_availability_candidates(None, "2026-05-30")
print("rooms:", len(BOOKING_DATA["rooms"]))
print("bookings:", len(BOOKING_DATA["bookings"]))
print("candidate_count_hint:", BOOKING_DATA["meta"]["candidate_count_hint"])
print("local_json_chars:", len(DATA_PATH.read_text(encoding="utf-8")))
print("candidate_count:", len(sample_candidates))
print("candidate_json_chars:", json_chars(sample_candidates))
print("JSON 中的脱敏样本:")
pprint(BOOKING_DATA["bookings"][0])
print("候选样本:")
pprint(sample_candidates[0])


## 4. 普通做法：直接定义 Tool 并调用

先不做裁剪，也不做 runtime 旁路透传。按照最普通的想法，把业务查询函数包装成 Tool，然后用 `.invoke()` 看返回效果。

这一段只做两件事：

1. 用 `@tool` 定义简单规则查询。
2. 用 `@tool(args_schema=...)` 定义带结构化参数的空位查询。

此时还不出现 `max_items_for_llm`、`runtime_state` 和模型配置。普通 Tool 能返回正确数据，再观察它放进 Agent 消息流后会发生什么。


In [ ]:
@tool
def query_room_rules(room_name: str) -> str:
    """查询书房预约规则。room_name 是书房名称，例如 平原轩 或 兰台。"""
    for room in BOOKING_DATA["rooms"]:
        if room["room_name"] == room_name:
            return f"{room_name} 预约规则：{room['rules']}"
    return f"未找到 {room_name} 的预约规则。"


class NaiveAvailabilityInput(BaseModel):
    room_name: str | None = Field(default=None, description="书房名称，例如 平原轩")
    booking_date: str = Field(description="预约日期，格式 YYYY-MM-DD")
    start_after: str | None = Field(default=None, description="只返回该时间之后的可用时段，格式 HH:MM")


@tool(args_schema=NaiveAvailabilityInput, return_direct=False)
def query_availability_naive(
    room_name: str | None,
    booking_date: str,
    start_after: str | None = None,
) -> str:
    """查询可预约候选，直接返回完整 JSON。"""
    items = collect_availability_candidates(room_name, booking_date, start_after)
    return json.dumps({"items": items}, ensure_ascii=False)


print("规则 Tool 名称:", query_room_rules.name)
print("规则 Tool 参数:", query_room_rules.args)
print(query_room_rules.invoke({"room_name": "平原轩"}))

naive_payload = query_availability_naive.invoke({
    "room_name": None,
    "booking_date": "2026-05-30",
    "start_after": None,
})
print("\n空位 Tool 名称:", query_availability_naive.name)
print("空位 Tool 参数:", query_availability_naive.args)
print("naive ToolMessage 字符数:", len(naive_payload))
print("naive 候选条数:", naive_payload.count("candidate_id"))
print(naive_payload[:500] + "\n...[后面还有大量 JSON]")


## 5. Tool 定义方式 1/2：`@tool` 与 `@tool(args_schema=...)`

上一个单元已经跑了两种最常用写法：

1. `@tool`：适合参数少、函数签名已经清晰的工具，例如 `query_room_rules`。
2. `@tool(args_schema=...)`：适合参数多、需要描述字段含义或约束的工具，例如 `query_availability_naive`。

这两种写法都会生成工具名称、描述和参数 schema。区别在于第二种 schema 更明确，模型更容易构造正确的 `tool_calls`。


In [ ]:
print("query_room_rules schema:")
pprint(query_room_rules.args)
print("\nquery_availability_naive schema:")
pprint(query_availability_naive.args)


## 6. 坏例子：普通 Tool 放进 Agent 消息流后撑爆上下文

`query_availability_naive.invoke()` 能返回正确数据，但它把完整候选 JSON 全部放进 `ToolMessage`。单独调用 Tool 时只是输出很长；放进 Agent 循环后，下一次模型调用会携带这段巨大消息。

下面先不修改 Tool，只把普通 Tool 放进最小 `ToolNode` 链路，观察消息流规模。


In [ ]:
print("单独 invoke 时，普通 Tool 已经返回完整 JSON。")
print("naive ToolMessage 字符数:", len(naive_payload))
print("naive 候选条数:", naive_payload.count("candidate_id"))
print("预览:")
print(naive_payload[:500] + "\n...[后面还有大量 JSON]")


## 7. 问题观察：Agent 运行到下一步前已经爆了

下面构造一个最小 Agent 消息流：用户消息 → 模型发起 `tool_calls` → `ToolNode` 执行未裁剪工具。此时还没进入下一次模型调用，消息流已经包含完整 `ToolMessage`。

这里设置一个字符预算来模拟上下文上限。普通工具返回超过预算后，流程会阻断下一次模型调用。


In [ ]:
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode

AGENT_CONTEXT_BUDGET_CHARS = 500_000


def message_content_chars(messages: list) -> int:
    return sum(len(getattr(message, "content", "") or "") for message in messages)


def build_availability_tool_args(max_items_for_llm: int | None = None) -> dict[str, Any]:
    args = {
        "room_name": None,
        "booking_date": "2026-05-30",
        "start_after": None,
    }
    if max_items_for_llm is not None:
        args["max_items_for_llm"] = max_items_for_llm
    return args


def run_tool_message_flow(
    tool_for_node,
    tool_name: str,
    runtime: dict,
    max_items_for_llm: int | None = None,
) -> dict[str, Any]:
    graph_builder = StateGraph(MessagesState)
    graph_builder.add_node("tools", ToolNode([tool_for_node]))
    graph_builder.add_edge(START, "tools")
    graph_builder.add_edge("tools", END)
    graph = graph_builder.compile()

    user_message = HumanMessage(content="帮我查 2026-05-30 所有书房有哪些空位，只需要先给我几个可选项。")
    tool_call_message = AIMessage(
        content="",
        tool_calls=[
            {
                "name": tool_name,
                "args": build_availability_tool_args(max_items_for_llm),
                "id": f"call_{tool_name}",
            }
        ],
    )

    tool_result = graph.invoke(
        {"messages": [user_message, tool_call_message]},
        config={"configurable": {"runtime_state": runtime}},
    )
    messages = tool_result["messages"]
    total_chars = message_content_chars(messages)
    context_exploded = total_chars > AGENT_CONTEXT_BUDGET_CHARS
    result = {
        "messages": messages,
        "message_count": len(messages),
        "tool_message_chars": [len(message.content) for message in messages if isinstance(message, ToolMessage)],
        "total_message_chars": total_chars,
        "context_budget_chars": AGENT_CONTEXT_BUDGET_CHARS,
        "context_exploded": context_exploded,
        "runtime_keys": sorted(runtime.keys()),
        "runtime_candidate_count": len(runtime.get("availability_raw", {}).get("items", [])),
    }
    if context_exploded:
        result["model_called"] = False
        result["blocked_reason"] = "工具结果进入 Agent 消息流后超过上下文预算，阻断下一次模型调用。"
    return result


naive_agent_runtime = {"user_id": "user_demo_001", "session_id": "session_naive_agent"}
naive_agent_flow = run_tool_message_flow(
    query_availability_naive,
    "query_availability_naive",
    naive_agent_runtime,
)
print("naive_agent_tool_message_chars:", naive_agent_flow["tool_message_chars"])
print("naive_agent_total_message_chars:", naive_agent_flow["total_message_chars"])
print("naive_context_exploded:", naive_agent_flow["context_exploded"])
print("naive_model_called:", naive_agent_flow["model_called"])
print("blocked_reason:", naive_agent_flow["blocked_reason"])


## 8. 处理方案：裁剪返回值 + runtime 旁路透传

普通工具已经把 Agent 消息流撑爆。现在再加入处理：

- `max_items_for_llm`：控制进入 LLM 上下文的候选数量。
- `clip_items_for_llm`：只让 LLM 看到少量摘要。
- `runtime_state_from_config`：完整候选通过运行时旁路保存。
- `build_llm`：从根目录 `.env` 读取 OpenAI 兼容模型配置。

从这一段开始，Tool 的返回值和完整业务数据分层处理：LLM 只读摘要，程序继续持有完整数据。


In [ ]:
class AvailabilityInput(BaseModel):
    room_name: str | None = Field(default=None, description="书房名称，例如 平原轩")
    booking_date: str = Field(description="预约日期，格式 YYYY-MM-DD")
    start_after: str | None = Field(default=None, description="只返回该时间之后的可用时段，格式 HH:MM")
    max_items_for_llm: int = Field(default=5, ge=1, le=20, description="返回给 LLM 的候选条数上限")


def clip_items_for_llm(items: list[dict], max_items: int = 5) -> tuple[list[dict], dict]:
    clipped = items[:max_items]
    return clipped, {
        "total_items": len(items),
        "llm_items": len(clipped),
        "raw_chars": json_chars(items),
        "llm_chars": json_chars(clipped),
        "truncated": len(items) > len(clipped),
    }


def runtime_state_from_config(config: RunnableConfig | None) -> dict:
    configurable = (config or {}).get("configurable", {})
    return configurable.setdefault("runtime_state", {})


load_dotenv(dotenv_path=Path.cwd() / ".env")
LLM_MODEL = os.getenv("OPENAI_MODEL", "deepseek-v4-flash")
LLM_BASE_URL = os.getenv("OPENAI_BASE_URL")
LLM_API_KEY = os.getenv("OPENAI_API_KEY")
LLM_TEMPERATURE = float(os.getenv("OPENAI_TEMPERATURE", "0.7"))
LLM_MAX_TOKENS = int(os.getenv("OPENAI_MAX_TOKENS", "128000"))


def build_llm():
    if not LLM_API_KEY:
        raise RuntimeError("未检测到 OPENAI_API_KEY。请先在项目根目录 .env 中配置模型密钥。")
    from langchain_openai import ChatOpenAI

    kwargs = {
        "model": LLM_MODEL,
        "temperature": LLM_TEMPERATURE,
        "max_tokens": LLM_MAX_TOKENS,
        "api_key": LLM_API_KEY,
    }
    if LLM_BASE_URL:
        kwargs["base_url"] = LLM_BASE_URL
    return ChatOpenAI(**kwargs)


@tool(args_schema=AvailabilityInput, return_direct=False)
def query_availability(
    room_name: str | None,
    booking_date: str,
    config: RunnableConfig,
    start_after: str | None = None,
    max_items_for_llm: int = 5,
) -> str:
    """返回可预约候选摘要，并把完整候选写入 runtime_state。"""
    items = collect_availability_candidates(room_name, booking_date, start_after)
    clipped, stats = clip_items_for_llm(items, max_items=max_items_for_llm)

    runtime_state = runtime_state_from_config(config)
    runtime_state["availability_raw"] = {
        "query": {
            "room_name": room_name,
            "booking_date": booking_date,
            "start_after": start_after,
        },
        "items": items,
    }
    runtime_state["availability_stats"] = stats

    if not items:
        return f"{room_name or '全部书房'} 在 {booking_date} 没有可预约时段。"

    lines = [
        f"{idx + 1}. {item['room_name']} {item['area_name']} {item['seat_name']} {item['start_time']}-{item['end_time']}"
        for idx, item in enumerate(clipped)
    ]
    more = "" if not stats["truncated"] else f"\n已裁剪：只给 LLM {stats['llm_items']} 条，完整 {stats['total_items']} 条已写入 runtime_state。"
    return f"{booking_date} 可预约候选摘要：\n" + "\n".join(lines) + more


runtime_state = {"user_id": "user_demo_001", "session_id": "session_demo_001"}
summary = query_availability.invoke(
    {
        "room_name": None,
        "booking_date": "2026-05-30",
        "start_after": None,
        "max_items_for_llm": 3,
    },
    config={"configurable": {"runtime_state": runtime_state}},
)
print("llm_model:", LLM_MODEL)
print("llm_base_url_configured:", bool(LLM_BASE_URL))
print("llm_api_key_configured:", bool(LLM_API_KEY))
print("安全 ToolMessage 字符数:", len(summary))
print(summary)
print("runtime_state keys:", runtime_state.keys())
print("stats:", runtime_state["availability_stats"])


## 9. Tool 定义方式 3：`StructuredTool.from_function` 与 `return_direct`

`StructuredTool.from_function` 可以把已有函数包装成 Tool，适合已有业务函数不想改装饰器的场景。

`return_direct` 是 Agent 路由信号，不只是 Tool 的一个属性：

1. `return_direct=False`：工具结果回到模型，模型读取 `ToolMessage` 后继续组织答复。
2. `return_direct=True`：工具结果作为终止信号，Agent 在工具执行后直接结束，不再回到模型。

下面用本地 fake model 构造固定的 `tool_calls`，直接观察 `create_agent` 在两种工具上的消息流差异。


In [ ]:
def calculate_booking_duration(start_time: str, end_time: str) -> str:
    """计算预约时长。"""
    start_hour = int(start_time.split(":")[0])
    end_hour = int(end_time.split(":")[0])
    return f"预约时长 {end_hour - start_hour} 小时"


duration_tool = StructuredTool.from_function(
    func=calculate_booking_duration,
    name="calculate_booking_duration",
    description="计算预约时长",
    return_direct=True,
    handle_tool_error=True,
)


def summarize_duration(start_time: str, end_time: str) -> str:
    """计算预约时长，结果交回模型继续总结。"""
    return calculate_booking_duration(start_time, end_time)


summary_duration_tool = StructuredTool.from_function(
    func=summarize_duration,
    name="summarize_duration",
    description="计算预约时长，结果回到模型继续总结",
    return_direct=False,
    handle_tool_error=True,
)


@tool(return_direct=True)
def calculate_booking_duration_direct(start_time: str, end_time: str) -> str:
    """计算预约时长，结果直接返回。"""
    return calculate_booking_duration(start_time, end_time)


print(duration_tool.invoke({"start_time": "16:00", "end_time": "18:00"}))
print("query_availability.return_direct =", query_availability.return_direct)
print("summary_duration_tool.return_direct =", summary_duration_tool.return_direct)
print("duration_tool.return_direct =", duration_tool.return_direct)
print("calculate_booking_duration_direct.return_direct =", calculate_booking_duration_direct.return_direct)

from langchain.agents import create_agent
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel


class ToolCallingFakeModel(FakeMessagesListChatModel):
    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        return self


def run_return_direct_agent_case(tool_for_agent, tool_name: str, direct: bool) -> dict[str, Any]:
    responses = [
        AIMessage(
            content="",
            tool_calls=[
                {
                    "name": tool_name,
                    "args": {"start_time": "16:00", "end_time": "18:00"},
                    "id": f"call_{tool_name}",
                }
            ],
        ),
        AIMessage(content="模型读取工具结果后生成：预约 2 小时，已继续整理成自然语言。"),
    ]
    fake_model = ToolCallingFakeModel(responses=responses)
    agent = create_agent(
        model=fake_model,
        tools=[tool_for_agent],
        system_prompt="你是书房预约助手。",
    )
    result = agent.invoke({"messages": [{"role": "user", "content": "帮我算 16:00 到 18:00 要预约多久"}]})
    messages = result["messages"]
    return {
        "tool_name": tool_name,
        "return_direct": direct,
        "message_types": [type(message).__name__ for message in messages],
        "message_count": len(messages),
        "last_message_type": type(messages[-1]).__name__,
        "last_message_content": messages[-1].content,
        "model_called_after_tool": isinstance(messages[-1], AIMessage),
    }


return_direct_results = {
    "false": run_return_direct_agent_case(summary_duration_tool, "summarize_duration", direct=False),
    "true": run_return_direct_agent_case(duration_tool, "calculate_booking_duration", direct=True),
}

print("\nreturn_direct=False 的 Agent 消息流:")
pprint(return_direct_results["false"])
print("\nreturn_direct=True 的 Agent 消息流:")
pprint(return_direct_results["true"])


## 10. Tool 定义方式 4：继承 `BaseTool`

工具需要独立类、复用内部方法、挂载更多元数据时，可以继承 `BaseTool`。下面的历史查询直接读取本地 JSON 里的脱敏记录，再裁剪返回给 LLM。

In [ ]:
class BookingHistoryInput(BaseModel):
    user_id: str = Field(description="用户 ID")
    max_items_for_llm: int = Field(default=3, ge=1, le=20, description="返回给 LLM 的历史记录条数上限")


class QueryBookingHistoryTool(BaseTool):
    name: str = "query_booking_history"
    description: str = "查询指定用户最近的书房预约记录，返回脱敏摘要，并把完整脱敏记录写入 runtime_state"
    args_schema: Type[BaseModel] = BookingHistoryInput

    def _run(
        self,
        user_id: str,
        config: RunnableConfig,
        max_items_for_llm: int = 3,
    ) -> str:
        records = [item for item in BOOKING_DATA["bookings"] if item["user_id"] == user_id]
        clipped, stats = clip_items_for_llm(records, max_items=max_items_for_llm)

        runtime_state = runtime_state_from_config(config)
        runtime_state["booking_history_redacted"] = records
        runtime_state["booking_history_stats"] = stats

        if not clipped:
            return "没有查到最近预约记录。"
        lines = [
            f"{idx + 1}. {item['room_name']} {item['area_name']} {item['seat_name']} {item['start_datetime']} - {item['status']}"
            for idx, item in enumerate(clipped)
        ]
        return "最近预约记录（已脱敏）：\n" + "\n".join(lines)


history_tool = QueryBookingHistoryTool()
history_runtime = {}
print(history_tool.invoke(
    {"user_id": "user_demo_001", "max_items_for_llm": 1},
    config={"configurable": {"runtime_state": history_runtime}},
))
history_records = history_runtime["booking_history_redacted"]
print("runtime 中保存的完整脱敏记录条数:", len(history_records))
print("runtime 中保存的完整脱敏记录字符数:", json_chars(history_records))
print("runtime 样本，只展示前 2 条:")
pprint(history_records[:2])

## 11. Tool 定义方式 5：运行时注入

生产项目里，用户 ID、会话 ID、权限、Store 等运行时数据不应该由 LLM 生成。LangGraph 可以用 `InjectedState / InjectedStore` 注入；本 notebook 用 `RunnableConfig.configurable.runtime_state` 演示同一类思路。


In [ ]:
from langgraph.prebuilt import InjectedState, InjectedStore
from langgraph.store.base import BaseStore


@tool
async def query_recent_preferences(
    room_name: str,
    state: Annotated[dict, InjectedState()],
    store: Annotated[BaseStore, InjectedStore()],
) -> str:
    """查询用户过去的预约偏好。state 和 store 由运行时注入，不由 LLM 传参。"""
    user_id = state.get("user_id", "unknown")
    memories = await store.asearch(("booking_preferences", user_id), query=room_name)
    return f"找到 {len(memories)} 条和 {room_name} 相关的预约偏好。"


print("InjectedState / InjectedStore 工具已定义。真实运行时由 LangGraph 注入 state 和 store。")


## 12. ToolNode：本地可运行的工具执行链路

正常 ReAct 流程中，LLM 会先产出 `tool_calls`，然后 `ToolNode` 负责真正执行工具。为了让本地环境不依赖 API key，这里手工构造一个带 `tool_calls` 的 `AIMessage`，再交给 `ToolNode` 执行。


In [ ]:
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode

booking_tools = [
    query_room_rules,
    query_availability,
    duration_tool,
    history_tool,
]

tool_node = ToolNode(booking_tools)

tool_graph_builder = StateGraph(MessagesState)
tool_graph_builder.add_node("tools", tool_node)
tool_graph_builder.add_edge(START, "tools")
tool_graph_builder.add_edge("tools", END)
tool_graph = tool_graph_builder.compile()

runtime_state = {"user_id": "user_demo_001", "session_id": "session_demo_001"}
manual_tool_call = AIMessage(
    content="",
    tool_calls=[
        {
            "name": "query_availability",
            "args": {
                "room_name": None,
                "booking_date": "2026-05-30",
                "start_after": None,
                "max_items_for_llm": 3,
            },
            "id": "call_availability_1",
        }
    ],
)

result = tool_graph.invoke(
    {"messages": [HumanMessage(content="查平原轩 15:00 之后的空位"), manual_tool_call]},
    config={"configurable": {"runtime_state": runtime_state}},
)

for message in result["messages"]:
    if isinstance(message, ToolMessage):
        print("ToolMessage content:\n", message.content)

print("\n旁路 runtime_state 中的完整数据统计:")
pprint(runtime_state["availability_stats"])
print("完整候选条数:", len(runtime_state["availability_raw"]["items"]))


## 13. 提交预约 Tool：用 runtime 中的候选继续执行

上一步没有把完整候选塞进 LLM 上下文，但候选已经进入 `runtime_state`。提交预约时，Tool 可以继续读取这份旁路数据做校验。


In [ ]:
class CreateBookingInput(BaseModel):
    candidate_id: str = Field(description="候选时段 ID，来自查询可预约时段的结果")


@tool(args_schema=CreateBookingInput, return_direct=False)
def create_booking(candidate_id: str, config: RunnableConfig) -> str:
    """根据候选 ID 提交书房预约。候选详情从 runtime_state 读取，不要求 LLM 重传完整对象。"""
    runtime_state = runtime_state_from_config(config)
    availability = runtime_state.get("availability_raw") or {}
    items = availability.get("items") or []
    selected = next((item for item in items if item["candidate_id"] == candidate_id), None)
    if selected is None:
        return "没有找到这个候选时段。请先重新查询可预约时段，再选择候选。"

    booking = {
        "booking_id": "booking_demo_003",
        "user_id": runtime_state.get("user_id", "unknown"),
        "room_name": selected["room_name"],
        "area_name": selected["area_name"],
        "seat_name": selected["seat_name"],
        "start_datetime": f"{selected['booking_date']} {selected['start_time']}:00",
        "end_datetime": f"{selected['booking_date']} {selected['end_time']}:00",
        "status": "submitted",
    }
    runtime_state["latest_booking"] = booking
    return (
        "预约已提交："
        f"{booking['room_name']} {booking['area_name']} {booking['seat_name']} "
        f"{booking['start_datetime']} - {booking['end_datetime']}。"
    )


booking_tool_graph_builder = StateGraph(MessagesState)
booking_tool_graph_builder.add_node("tools", ToolNode([create_booking]))
booking_tool_graph_builder.add_edge(START, "tools")
booking_tool_graph_builder.add_edge("tools", END)
booking_tool_graph = booking_tool_graph_builder.compile()

first_candidate = runtime_state["availability_raw"]["items"][0]
booking_call = AIMessage(
    content="",
    tool_calls=[
        {
            "name": "create_booking",
            "args": {"candidate_id": first_candidate["candidate_id"]},
            "id": "call_booking_1",
        }
    ],
)
booking_result = booking_tool_graph.invoke(
    {"messages": [booking_call]},
    config={"configurable": {"runtime_state": runtime_state}},
)

for message in booking_result["messages"]:
    if isinstance(message, ToolMessage):
        print(message.content)

print("\nruntime_state.latest_booking:")
pprint(runtime_state["latest_booking"])


## 14. Agent 运行链路：安全工具可以继续进入模型

现在再用同样的 Agent 消息流执行安全工具。ToolMessage 很小，完整候选留在 `runtime_state`。如果 `.env` 中配置了模型，安全链路会继续调用模型生成最终答复。

In [ ]:
agent_results: dict[str, Any] = {"naive": naive_agent_flow}


def run_safe_agent_loop(runtime: dict, max_items_for_llm: int = 3) -> dict[str, Any]:
    flow = run_tool_message_flow(
        query_availability,
        "query_availability",
        runtime,
        max_items_for_llm=max_items_for_llm,
    )
    result = {
        key: value
        for key, value in flow.items()
        if key != "messages"
    }

    if result["context_exploded"]:
        result["model_called"] = False
        result["blocked_reason"] = "工具结果进入 Agent 消息流后超过上下文预算，阻断下一次模型调用。"
        return result

    if not LLM_API_KEY:
        result["model_called"] = False
        result["blocked_reason"] = "未检测到 OPENAI_API_KEY，跳过模型调用。"
        return result

    llm = build_llm()
    final_answer = llm.invoke(flow["messages"])
    result["model_called"] = True
    result["final_answer"] = final_answer.content
    return result


safe_agent_runtime = {"user_id": "user_demo_001", "session_id": "session_safe_agent"}
agent_results["safe"] = run_safe_agent_loop(safe_agent_runtime)

print("naive Agent 消息流结果:")
pprint({key: value for key, value in agent_results["naive"].items() if key != "messages"})
print("\nsafe Agent 消息流结果:")
pprint(agent_results["safe"])

## 15. `llm.bind_tools`：只绑定，不执行

`bind_tools` 让模型知道有哪些工具，并输出 `tool_calls`。它不会替你执行工具。执行环节仍然需要 `ToolNode`、`create_agent` 或自己写编排逻辑。

In [ ]:
if LLM_API_KEY:
    llm = build_llm()
    llm_with_tools = llm.bind_tools([query_room_rules, query_availability, history_tool])
    ai_message = llm_with_tools.invoke("查一下平原轩 2026-05-30 15:00 之后有哪些空位")
    print("model content:", ai_message.content)
    print("tool_calls:")
    pprint(ai_message.tool_calls)
else:
    print("未检测到 OPENAI_API_KEY，跳过 bind_tools 调用。")

## 16. 缓存：哪些数据可以缓存

书房规则变化不频繁，可以缓存。实时预约提交结果不能缓存。当天空位可以短 TTL 缓存，但提交前还要重新校验。


In [ ]:
from functools import lru_cache


@lru_cache(maxsize=128)
def query_room_rules_cached_plain(room_name: str) -> str:
    for room in BOOKING_DATA["rooms"]:
        if room["room_name"] == room_name:
            return room["rules"]
    return "未找到规则"


query_room_rules_cached_plain.cache_clear()
first_rules = query_room_rules_cached_plain("平原轩")
first_cache_info = query_room_rules_cached_plain.cache_info()
second_rules = query_room_rules_cached_plain("平原轩")
second_cache_info = query_room_rules_cached_plain.cache_info()

print("第一次查规则:", first_rules)
print("第一次 cache_info:", first_cache_info)
print("第二次查同一个书房:", second_rules)
print("第二次 cache_info:", second_cache_info)
print("缓存命中次数变化:", second_cache_info.hits - first_cache_info.hits)


## 17. 上下文管理：最终规则

Tool 返回给 LLM 的内容遵守三条规则：

1. 只返回摘要、前几条候选和下一步引导。
2. 数据在进入本地 JSON 前完成脱敏，案例代码只消费脱敏后的数据。
3. 完整结构化数据写入 runtime、State 或 Store，给后续节点继续使用。

最终演示会同时展示三层差异：ToolMessage 的规模差异，真实 Agent 消息流在“未裁剪 / 已裁剪”下的运行差异，以及 `return_direct` 对 Agent 是否回到模型的影响。

In [ ]:
raw_items = runtime_state["availability_raw"]["items"]
tool_message = next(message for message in result["messages"] if isinstance(message, ToolMessage))
history_records = history_runtime["booking_history_redacted"]
history_summary = history_tool.invoke(
    {"user_id": "user_demo_001", "max_items_for_llm": 1},
    config={"configurable": {"runtime_state": {}}},
)
cache_delta = second_cache_info.hits - first_cache_info.hits

print("=== 最终演示：Tool 返回值上下文爆炸 ===")
print("naive ToolMessage 字符数:", len(naive_payload))
print("naive 候选条数:", naive_payload.count("candidate_id"))
print("安全 ToolMessage 字符数:", len(tool_message.content))
print("安全裁剪比例:", f"{len(tool_message.content) / len(naive_payload):.4%}")

print("\n=== 最终演示：Agent 未裁剪 vs 已裁剪 ===")
naive_agent_result = agent_results["naive"]
safe_agent_result = agent_results["safe"]
print("naive_context_exploded:", naive_agent_result["context_exploded"])
print("naive_model_called:", naive_agent_result["model_called"])
print("naive_tool_message_chars:", naive_agent_result["tool_message_chars"])
print("naive_total_message_chars:", naive_agent_result["total_message_chars"])
print("safe_context_exploded:", safe_agent_result["context_exploded"])
print("safe_model_called:", safe_agent_result["model_called"])
print("safe_tool_message_chars:", safe_agent_result["tool_message_chars"])
print("safe_total_message_chars:", safe_agent_result["total_message_chars"])
print("safe_runtime_candidate_count:", safe_agent_result["runtime_candidate_count"])
if safe_agent_result.get("final_answer"):
    print("safe_final_answer:", safe_agent_result["final_answer"])
else:
    print("safe_blocked_reason:", safe_agent_result.get("blocked_reason"))

print("\n=== 最终演示：return_direct 的 Agent 路由 ===")
return_false = return_direct_results["false"]
return_true = return_direct_results["true"]
print("return_direct=False message_types:", return_false["message_types"])
print("return_direct=False model_called_after_tool:", return_false["model_called_after_tool"])
print("return_direct=False last_message_type:", return_false["last_message_type"])
print("return_direct=True message_types:", return_true["message_types"])
print("return_direct=True model_called_after_tool:", return_true["model_called_after_tool"])
print("return_direct=True last_message_type:", return_true["last_message_type"])
print("return_direct=True final_content:", return_true["last_message_content"])

print("\n=== 最终演示：runtime 透传 ===")
print("runtime_state keys:", sorted(runtime_state.keys()))
print("availability 是否透传成功:", "availability_raw" in runtime_state)
print("runtime 完整候选条数:", len(raw_items))
print("runtime 完整候选 JSON 字符数:", json_chars(raw_items))
print("提交预约读取 runtime 后生成 latest_booking:")
pprint(runtime_state["latest_booking"])
print("历史 ToolMessage 字符数:", len(history_summary))
print("history_runtime keys:", sorted(history_runtime.keys()))
print("history 是否透传成功:", "booking_history_redacted" in history_runtime)
print("runtime 完整历史条数:", len(history_records))
print("runtime 完整历史 JSON 字符数:", json_chars(history_records))

print("\n=== 最终演示：缓存命中 ===")
print("第一次 cache_info:", first_cache_info)
print("第二次 cache_info:", second_cache_info)
print("第二次是否命中缓存:", cache_delta == 1)

## 18. 练习

1. 把 `query_availability` 的裁剪策略从固定条数改成按字符数裁剪。
2. 给可预约时段增加短 TTL 缓存，并在提交预约前重新校验。
3. 增加一个 `cancel_booking` Tool，要求返回可恢复错误。
4. 用 `create_agent` 跑一次完整对话：查询空位 → 选择候选 → 提交预约。
5. 把 `runtime_state` 换成 `InjectedStore`，模拟跨会话复用。

---
**下一节**: LG-03 Human-in-the-loop
